# Exploratory Data Analysis — Comprehensive EDA

This notebook performs a complete exploratory data analysis on the churn prediction dataset. We'll validate data quality, screen features statistically, check for temporal leakage, and create analytical tables to understand churn patterns.

## Section 0 · Setup & Configuration

First thing to do is bring in the config setup. This loads all our helper functions and constants like table names and the churn threshold. It's the same config file the main notebooks use, so everything stays consistent.

In [0]:
%run /Users/vidyutparvat2@gmail.com/ds-mini-project/02_config/config_setup

Now let's import the libraries we'll need for analysis. Pandas and numpy for data manipulation, matplotlib and seaborn for visualizations if we need them, and scipy for the statistical tests we'll run during feature screening.

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency, pointbiserialr

print("✓ Libraries imported successfully")

## Section 1 · Data Loading & Overview

Let's load all four tables from the catalog. The load_table helper function is already defined in the config, so we just call it for each table. We'll also parse the date columns in billings right away since we need those for the reference date calculation later.

In [0]:
section("Loading tables")
billings      = load_table(BILLINGS_TABLE,      BILLINGS_TABLE)
cc_calls      = load_table(CC_CALLS_TABLE,      CC_CALLS_TABLE)
emails        = load_table(EMAILS_TABLE,        EMAILS_TABLE)
renewal_calls = load_table(RENEWAL_CALLS_TABLE, RENEWAL_CALLS_TABLE)

billings = parse_dates(billings, ['Prospect_Renewal_Date', 'Closed_Date'])

print(f"\n✓ All tables loaded successfully")
print(f"  Billings      : {billings.shape[0]:,} rows × {billings.shape[1]} columns")
print(f"  CC Calls      : {cc_calls.shape[0]:,} rows × {cc_calls.shape[1]} columns")
print(f"  Emails        : {emails.shape[0]:,} rows × {emails.shape[1]} columns")
print(f"  Renewal Calls : {renewal_calls.shape[0]:,} rows × {renewal_calls.shape[1]} columns")

Now let's take a peek at what's in each table. Starting with billings since that's our main table with the customer outcomes.

In [0]:
section("Billings Table — Preview")
print("First 5 rows:")
display(billings.head())

print("\nData types:")
print(billings.dtypes)

print("\nMissing values:")
missing = billings.isnull().sum()
missing_pct = (missing / len(billings) * 100).round(1)
missing_df = pd.DataFrame({'Missing': missing, 'Percent': missing_pct})
missing_df = missing_df[missing_df['Missing'] > 0].sort_values('Missing', ascending=False)
print(missing_df.to_string())

print("\nKey ID column uniqueness:")
print(f"  Unique Co_Ref      : {billings['Co_Ref'].nunique():,}")
print(f"  Unique Customer_Ref: {billings['Customer_Ref'].nunique():,}")
print(f"  Total rows         : {len(billings):,}")

Now checking the CC calls table. This has info from customer care interactions.

In [0]:
section("CC Calls Table — Preview")
print("First 5 rows:")
display(cc_calls.head())

print("\nData types:")
print(cc_calls.dtypes.head(20))  # Just first 20 since this table has many columns

print("\nMissing values (top 10 columns):")
missing = cc_calls.isnull().sum()
missing_pct = (missing / len(cc_calls) * 100).round(1)
missing_df = pd.DataFrame({'Missing': missing, 'Percent': missing_pct})
missing_df = missing_df[missing_df['Missing'] > 0].sort_values('Missing', ascending=False).head(10)
print(missing_df.to_string())

print("\nKey ID column uniqueness:")
print(f"  Unique Co_Ref      : {cc_calls['Co_Ref'].nunique():,}")
print(f"  Total rows         : {len(cc_calls):,}")

Looking at emails now. This table doesn't have dates but it has a Time_to_Renewal bucket that tells us proximity to renewal.

In [0]:
section("Emails Table — Preview")
print("First 5 rows:")
display(emails.head())

print("\nData types:")
print(emails.dtypes.head(20))

print("\nMissing values (top 10 columns):")
missing = emails.isnull().sum()
missing_pct = (missing / len(emails) * 100).round(1)
missing_df = pd.DataFrame({'Missing': missing, 'Percent': missing_pct})
missing_df = missing_df[missing_df['Missing'] > 0].sort_values('Missing', ascending=False).head(10)
print(missing_df.to_string())

print("\nTime_to_Renewal bucket distribution:")
if 'Time_to_Renewal' in emails.columns:
    print(emails['Time_to_Renewal'].value_counts().to_string())

print("\nKey ID column uniqueness:")
print(f"  Unique Co_Ref      : {emails['Co_Ref'].nunique():,}")
print(f"  Total rows         : {len(emails):,}")

Finally, renewal calls. These are specific calls about renewals, different from the general customer care calls.

In [0]:
section("Renewal Calls Table — Preview")
print("First 5 rows:")
display(renewal_calls.head())

print("\nData types:")
print(renewal_calls.dtypes)

print("\nMissing values (top 10 columns):")
missing = renewal_calls.isnull().sum()
missing_pct = (missing / len(renewal_calls) * 100).round(1)
missing_df = pd.DataFrame({'Missing': missing, 'Percent': missing_pct})
missing_df = missing_df[missing_df['Missing'] > 0].sort_values('Missing', ascending=False).head(10)
print(missing_df.to_string())

print("\nKey ID column uniqueness:")
print(f"  Unique Co_Ref      : {renewal_calls['Co_Ref'].nunique():,}")
print(f"  Total rows         : {len(renewal_calls):,}")

## Section 2 · Billings — Deduplication & Reference Date

The billings table has multiple rows per Co_Ref because customers renew multiple times. For churn modeling, we want the first renewal for each customer. So we're deduplicating by Co_Ref and keeping the oldest record based on Prospect_Renewal_Date.

In [0]:
section("Billings — Co_Ref deduplication: keep oldest record")
before = len(billings)
billings = (billings
            .sort_values('Prospect_Renewal_Date', ascending=True)
            .drop_duplicates(subset='Co_Ref', keep='first'))
after = len(billings)
print(f"  Before : {before:,} rows")
print(f"  After  : {after:,} unique Co_Ref")
print(f"  Removed: {before - after:,} duplicate Co_Ref rows (non-oldest records dropped)")

Now we need to compute a reference date for each customer. This is the date we'll use when linking calls and emails — we want interactions that happened before this date, not after (to avoid leakage). If the customer has a Closed_Date, we use that. Otherwise we add 28 days to Prospect_Renewal_Date as a proxy.

In [0]:
section("Billings — Compute reference_date per customer")
billings['days_to_close'] = (
    billings['Closed_Date'] - billings['Prospect_Renewal_Date']).dt.days

# reference_date = Closed_Date if available, else Prospect_Renewal_Date + 28 days
billings['reference_date'] = billings['Closed_Date'].fillna(
    billings['Prospect_Renewal_Date'] + pd.Timedelta(days=28))

print(f"reference_date source breakdown:")
print(f"  Using Closed_Date          : {billings['Closed_Date'].notna().sum():,} customers ({billings['Closed_Date'].notna().mean()*100:.1f}%)")
print(f"  Using PRD + 28 days (no close): {billings['Closed_Date'].isna().sum():,} customers ({billings['Closed_Date'].isna().mean()*100:.1f}%)")
print(f"  Null reference_date        : {billings['reference_date'].isna().sum()}")

## Section 3 · Churn Labeling & Target Distribution

Time to create our target variable. We have a Prospect_Outcome column that tells us if someone churned, won, or is still open. But we need to handle some edge cases:
- If someone churned but has no Closed_Date, that's a confirmed churn
- If they churned but it took more than the threshold days (defined in config), that's also a churn
- If days_to_close is negative, that's weird data quality, we'll call those "Pre_Churned" and exclude them
- Everyone else with Won or Open gets labeled accordingly

In [0]:
section("Billings — Apply churn labels")
conditions = [
    (billings['Prospect_Outcome'] == 'Churned') & (billings['Closed_Date'].isna()),
    (billings['Prospect_Outcome'] == 'Churned') & (billings['days_to_close'] > CHURN_DAYS_THRESHOLD),
    (billings['Prospect_Outcome'] == 'Churned') & (billings['days_to_close'] < 0),
    billings['Prospect_Outcome'] == 'Won',
    billings['Prospect_Outcome'] == 'Open',
]
billings['Churn_Label'] = np.select(conditions,
    ['Churned', 'Churned', 'Pre_Churned', 'Won', 'Open'], default='Churned')

model_df = billings[billings['Churn_Label'].isin(['Won', 'Churned'])].copy()
model_df['Target'] = (model_df['Churn_Label'] == 'Churned').astype(int)

print(f"model_df: {len(model_df):,} rows  |  churn rate: {model_df['Target'].mean()*100:.2f}%")
label_counts = billings['Churn_Label'].value_counts()
for label, count in label_counts.items():
    print(f"  {label:<15}: {count:,}")

Let's also check if there's any year-based pattern in the churn labels. This helps us understand if churn is concentrated in certain years or pretty stable over time.

In [0]:
section("Churn Labels — Year-wise breakdown")
if 'Prospect_Renewal_Date' in billings.columns:
    billings['Renewal_Year'] = billings['Prospect_Renewal_Date'].dt.year
    year_breakdown = pd.crosstab(billings['Renewal_Year'], billings['Churn_Label'], margins=True)
    print("\nChurn labels by Prospect_Renewal_Date year:")
    display(year_breakdown)
    
    # Calculate churn rate by year for Won+Churned only
    year_churn = billings[billings['Churn_Label'].isin(['Won', 'Churned'])].groupby('Renewal_Year').agg(
        Total=('Churn_Label', 'count'),
        Churned=('Churn_Label', lambda x: (x == 'Churned').sum()),
        Churn_Rate=('Churn_Label', lambda x: (x == 'Churned').mean() * 100)
    ).round(2)
    print("\nChurn rate by year (Won + Churned only):")
    display(year_churn)
else:
    print("Prospect_Renewal_Date not available for year-wise breakdown")

## Section 4 · Statistical Screening Helpers

Before we dive into feature screening, we need some helper functions to test statistical significance. These functions will help us decide which features are actually predictive of churn and which ones we should drop.

The binary screening function uses chi-square tests to see if a Yes/No feature is associated with churn. The numeric screening function uses point-biserial correlation to measure the relationship between a continuous feature and our binary target.

We'll drop features if they have too many missing values (>=50%), aren't statistically significant (p >= 0.05), or show the wrong direction (higher values associated with lower churn, which doesn't make business sense).

In [0]:
def screen_binary_col(df, col, target_col='Target', yes_val='Yes'):
    """Screen a binary column for churn prediction significance.
    
    Returns: null_pct, chi2, p_value, churn_rate_yes%, churn_rate_no%
    """
    null_pct = df[col].isna().mean() * 100
    sub      = df[[col, target_col]].dropna()
    sub_bin  = (sub[col] == yes_val).astype(int)
    if sub_bin.nunique() < 2:
        return null_pct, np.nan, np.nan, np.nan, np.nan
    ct = pd.crosstab(sub_bin, sub[target_col])
    chi2, p, _, _ = chi2_contingency(ct)
    cr_yes = sub[sub_bin == 1][target_col].mean() * 100
    cr_no  = sub[sub_bin == 0][target_col].mean() * 100
    return null_pct, round(chi2, 1), round(p, 8), round(cr_yes, 2), round(cr_no, 2)

def screen_numeric_col(df, col, target_col='Target'):
    """Screen a numeric column for churn prediction significance.
    
    Returns: null_pct, correlation_r, p_value
    """
    null_pct = df[col].isna().mean() * 100
    series   = pd.to_numeric(df[col], errors='coerce')
    sub      = pd.DataFrame({'x': series, 'y': df[target_col]}).dropna()
    if len(sub) < 10:
        return null_pct, np.nan, np.nan
    r, p = pointbiserialr(sub['y'], sub['x'])
    return null_pct, round(r, 4), round(p, 8)

def screen_categorical_col(df, col, target_col='Target'):
    """Screen a categorical column for churn prediction significance.
    
    Returns: null_pct, chi2, p_value, cramers_v
    """
    null_pct = df[col].isna().mean() * 100
    sub      = df[[col, target_col]].dropna()
    ct       = pd.crosstab(sub[col], sub[target_col])
    chi2, p, _, _ = chi2_contingency(ct)
    n        = ct.values.sum()
    cramers  = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))
    return null_pct, round(chi2, 1), round(p, 8), round(cramers, 4)

# Thresholds for feature screening
P_THRESHOLD   = 0.05
NULL_DROP_PCT = 50.0

print("[EDA] Screening functions registered")
print(f"  Drop rules: null% >= {NULL_DROP_PCT}%  |  p >= {P_THRESHOLD}  |  wrong direction  |  leakage")

## Section 5 · Billings Table — Feature Screening

Now we'll run statistical tests on all the numeric features in billings to see which ones are actually predictive of churn. We're using point-biserial correlation here since we have continuous features and a binary target.

In [0]:
section("BILLINGS — Numeric Column Screening")
BILL_NUM_CANDIDATES = [
    'Auto_Renewal_Score', 'Status_Scores', 'Anchoring_Score', 'Tenure_Scores',
    'Tenure_Years', 'Current_Anchorings', 'Last_Years_Price', 'Starting_Net',
    'Total_Renewal_Score_New', 'Sustainability_Score', 'Renewal_Score_At_Release',
    'Connection_Net', 'Discount_Amount', 'Proforma_Approved_Lists', 'Payment_Timeframe',
]
bill_num_rows = []
for c in BILL_NUM_CANDIDATES:
    if c not in model_df.columns:
        continue
    model_df[c] = pd.to_numeric(model_df[c], errors='coerce')
    null_pct, r, p = screen_numeric_col(model_df, c)
    if null_pct >= NULL_DROP_PCT:
        decision = f'DROP — null%={null_pct:.1f}% exceeds {NULL_DROP_PCT}% threshold'
    elif np.isnan(p) or p >= P_THRESHOLD:
        decision = f'DROP — p={p:.3f} not significant'
    else:
        decision = f'KEEP — r={r:+.3f}, p={p:.2e}'
    bill_num_rows.append({'Column': c, 'Null%': round(null_pct, 1),
                          'r': r, 'p_value': p, 'Decision': decision})
bill_num_df = pd.DataFrame(bill_num_rows)
display_df(bill_num_df, "Billings Numeric — Statistical Screening")

Same thing for categorical features, but now we use chi-square tests and Cramér's V to measure association strength between categories and churn.

In [0]:
section("BILLINGS — Categorical Column Screening")
BILL_CAT_CANDIDATES = [
    'Band', 'Proforma_Account_Stage', 'Proforma_Membership_Status',
    'Payment_Method', 'Proforma_Audit_Status', 'Tenure_Group', 'Connection_Group',
]
bill_cat_rows = []
for c in BILL_CAT_CANDIDATES:
    if c not in model_df.columns:
        continue
    null_pct, chi2, p, cramers = screen_categorical_col(model_df, c)
    if null_pct >= NULL_DROP_PCT:
        decision = f'DROP — null%={null_pct:.1f}% exceeds {NULL_DROP_PCT}% threshold'
    elif np.isnan(p) or p >= P_THRESHOLD:
        decision = f'DROP — p={p:.3f} not significant'
    else:
        decision = f"KEEP — chi2={chi2:.0f}, p={p:.2e}, Cramér's V={cramers:.3f}"
    bill_cat_rows.append({'Column': c, 'Null%': round(null_pct, 1),
                          'chi2': chi2, 'p_value': p,
                          "Cramér's V": cramers, 'Decision': decision})
bill_cat_df = pd.DataFrame(bill_cat_rows)
display_df(bill_cat_df, "Billings Categorical — Statistical Screening")

Before we finalize the feature list, we need to explicitly identify leakage columns. These are columns that either directly contain the target or are computed after the outcome is known. Even if they're statistically significant, we can't use them for prediction.

In [0]:
section("BILLINGS — Leakage Columns (excluded regardless of significance)")
LEAKAGE_COLS = {
    'Total_Net_Paid'   : 'POST-OUTCOME: 0 for churned, actual amount for won — direct leakage',
    'Gross'            : 'POST-OUTCOME: derived from Total_Net_Paid',
    'Amount'           : 'POST-OUTCOME: actual payment amount',
    'Closed_Date'      : 'USED TO DERIVE TARGET — not a predictor',
    'reference_date'   : 'DERIVED FROM CLOSED_DATE — not a predictor',
    'days_to_close'    : 'DERIVED FROM CLOSED_DATE — leakage',
    'Prospect_Outcome' : 'IS THE TARGET — must not be a feature',
    'Prospect_Status'  : 'DERIVED FROM OUTCOME — leakage',
    'Churn_Label'      : 'DERIVED TARGET COLUMN — leakage',
}
leak_df = pd.DataFrame({'Column': list(LEAKAGE_COLS.keys()),
                        'Reason': list(LEAKAGE_COLS.values())})
display_df(leak_df, "Billings — Leakage Columns (all excluded)")

BILL_NUM_KEEP = bill_num_df[bill_num_df['Decision'].str.startswith('KEEP')]['Column'].tolist()
BILL_CAT_KEEP = bill_cat_df[bill_cat_df['Decision'].str.startswith('KEEP')]['Column'].tolist()

print("\nBilling numeric features kept  :", BILL_NUM_KEEP)
print("Billing categorical features kept:", BILL_CAT_KEEP)

## Section 6 · Emails Table — Proximity Validation & Feature Screening

Emails don't have actual dates, just a Time_to_Renewal bucket that tells us how close to renewal the email was sent. We need to validate that pre-churned accounts aren't concentrated in any particular bucket, which would indicate data quality issues.

In [0]:
section("EMAILS — Pre-churned contamination check across Time_to_Renewal buckets")
email_full = emails.merge(
    billings[['Co_Ref', 'Churn_Label']], on='Co_Ref', how='left')

ct = pd.crosstab(email_full['Time_to_Renewal'], email_full['Churn_Label'])
ct['Total'] = ct.sum(axis=1)
if 'Pre_Churned' in ct.columns:
    ct['Pre_Churned_%'] = (ct['Pre_Churned'] / ct['Total'] * 100).round(2)
display_df(ct.reset_index(), "Email rows by Time_to_Renewal bucket and Churn_Label")
print()
print("FINDING: Pre-churned accounts should be roughly evenly distributed across buckets.")
print("No bucket should be disproportionately contaminated.")
print("Pre-churned rows drop out naturally when merging with model_df (Won+Churned only).")

Now let's validate that the Time_to_Renewal ordering actually makes sense. The pre_renewal bucket should have the strongest signal (highest chi2) since it's closest to the renewal decision. If prior_year has stronger signal than pre_renewal, something's wrong with our proximity assumption.

In [0]:
section("EMAILS — Proximity ordering validation: chi2 by TTR bucket")
em_merged_all = emails.merge(model_df[['Co_Ref', 'Target']], on='Co_Ref', how='inner')
print("Chi2 for crm_contractor_suggested_leave by Time_to_Renewal bucket:")
print("(Higher chi2 = bucket carries stronger signal for that feature)\n")
for bucket in ['pre_renewal', '14_out', '45_out', 'prior_year']:
    sub = em_merged_all[em_merged_all['Time_to_Renewal'] == bucket].copy()
    sub['flag'] = (sub['crm_contractor_suggested_leave'] == 'Yes').astype(int)
    if sub['flag'].nunique() < 2 or len(sub) < 20:
        continue
    ct2 = pd.crosstab(sub['flag'], sub['Target'])
    chi2, p, _, _ = chi2_contingency(ct2)
    cr_yes = sub[sub['flag']==1]['Target'].mean()*100 if sub['flag'].sum()>0 else 0
    print(f"  {bucket:<15}: n={len(sub):>6,}  chi2={chi2:8.1f}  p={p:.3e}  "
          f"churn_when_flagged={cr_yes:.1f}%")
print()
print("CONCLUSION: pre_renewal should have the highest chi2 — it's the strongest signal bucket.")
print("TTR ordering (pre_renewal=1 → prior_year=4) is therefore the correct proximity proxy.")
print("Emails have no date column, so this ordering is the only available approximation.")

For email feature screening, we need to pick one email per customer — the one from the nearest TTR bucket. Then we run our statistical tests on that subset.

In [0]:
section("EMAILS — Screening population: nearest TTR bucket per customer")
TTR_ORDER = {'pre_renewal': 1, '14_out': 2, '45_out': 3, 'prior_year': 4}
emails['_ttr_rank'] = emails['Time_to_Renewal'].map(TTR_ORDER).fillna(5)
em_recent = (emails.sort_values(['Co_Ref', '_ttr_rank'])
                   .drop_duplicates('Co_Ref', keep='first'))
em_screen = em_recent.merge(model_df[['Co_Ref', 'Target']], on='Co_Ref', how='inner')
print(f"Screening population: {len(em_screen):,} unique customers")
print(f"Churn rate          : {em_screen['Target'].mean()*100:.2f}%")
print("\nBucket distribution of selected emails:")
print(em_screen['Time_to_Renewal'].value_counts().to_string())

Now screening the email binary features. These are Yes/No flags from CRM indicating things like complaints, switching intent, refund requests, etc.

In [0]:
EMAIL_BIN_CANDIDATES = [
    'crm_delays_in_accreditation', 'crm_contractor_suggested_leave',
    'crm_competitors_mentioned', 'crm_platform_issues_raised',
    'crm_customer_complained', 'crm_refund_mentioned',
    'crm_negative_customer_experience', 'crm_dissatisfaction_with_support',
    'crm_financial_hardship_mentioned', 'crm_agent_chased_contractor',
    'crm_dissatisified_with_renewal_price', 'crm_accreditation_issues',
]
em_bin_rows = []
for c in EMAIL_BIN_CANDIDATES:
    if c not in em_screen.columns:
        continue
    null_pct, chi2, p, cr_yes, cr_no = screen_binary_col(em_screen, c)
    if null_pct >= NULL_DROP_PCT:
        decision = f'DROP — null%={null_pct:.1f}% exceeds {NULL_DROP_PCT}% threshold'
    elif np.isnan(p) or p >= P_THRESHOLD:
        decision = f'DROP — p={p:.3f} not significant'
    elif cr_yes < cr_no:
        decision = f'DROP — wrong direction: churn_yes={cr_yes}% < churn_no={cr_no}%'
    else:
        decision = f'KEEP — chi2={chi2:.0f}, p={p:.2e}, churn_yes={cr_yes}% vs no={cr_no}%'
    em_bin_rows.append({'Column': c, 'Null%': round(null_pct,1),
                        'chi2': chi2, 'p_value': p,
                        'Churn_Yes%': cr_yes, 'Churn_No%': cr_no,
                        'Decision': decision})
em_bin_df = pd.DataFrame(em_bin_rows)
display_df(em_bin_df, "Emails Binary — Statistical Screening")